# π0.5 파인튜닝: AIRLAB 집기

노트북에서 만든 데이터셋으로 openpi의 π0.5를 파인튜닝하고, MuJoCo에서 평가합니다. 자세한 설명은 [pi05/README.md](https://github.com/sejiny3/AIRLAB/blob/main/pi05/README.md)에 있습니다.

**시작 전에**

1. 메뉴에서 **런타임 > 런타임 유형 변경 > A100 GPU**를 고릅니다. L4도 되지만 배치를 줄여서 돌립니다.
2. Google Drive의 `내 드라이브/airlab_pi05/`에 `pick_pi05.tar`가 있어야 합니다.

위에서부터 한 칸씩 실행합니다. 런타임이 끊겨도 체크포인트는 Drive에 남으니, 처음부터 다시 실행하면 이어서 학습합니다.

## 1. GPU와 Drive 확인

In [ ]:
import os, shutil, subprocess
from google.colab import drive

drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/airlab_pi05'

gpu, mem = subprocess.check_output(
    ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader,nounits'], text=True).strip().split(', ')
BATCH = 32 if int(mem) > 30000 else 16  # LoRA는 배치 32에서 22.5 GB 넘게 씁니다
free = shutil.disk_usage('/content/drive').free / 1e9
print(f'GPU {gpu} {int(mem) / 1024:.0f} GB, 배치 {BATCH}')
print(f'Drive 여유 공간 {free:.1f} GB (체크포인트 하나에 약 11 GB 필요)')
assert os.path.exists(f'{DRIVE}/pick_pi05.tar'), 'pick_pi05.tar가 Drive의 airlab_pi05 폴더에 없습니다'
assert free > 12, 'Drive 공간이 부족합니다'
os.environ.update(DRIVE=DRIVE, BATCH=str(BATCH))

## 2. 설치

AIRLAB, OpenArm 모델, openpi를 받고 파이썬 환경을 만듭니다. 3~5분 걸립니다. 마지막에 MuJoCo 렌더링까지 확인하므로, 학습에 GPU 시간을 쓰기 전에 문제를 알 수 있습니다.

`uv run`을 쓰면 MuJoCo가 openpi의 고정 버전으로 돌아가므로, 이 노트북은 `.venv/bin/python`을 직접 실행합니다.

In [ ]:
%%bash
set -e
cd /content
curl -LsSf https://astral.sh/uv/install.sh | sh > /dev/null
export PATH=$HOME/.local/bin:$PATH UV_HTTP_TIMEOUT=600 GIT_LFS_SKIP_SMUDGE=1

[ -d AIRLAB ] || git clone -q https://github.com/sejiny3/AIRLAB.git
git -C AIRLAB pull -q
if [ ! -d AIRLAB/third_party/openarm_mujoco ]; then
  git clone -q https://github.com/enactic/openarm_mujoco.git AIRLAB/third_party/openarm_mujoco
  git -C AIRLAB/third_party/openarm_mujoco checkout -q a3c8a57ec15d385e3c1dbd3fd090ee486b7f258c
fi
if [ ! -d openpi ]; then
  git clone -q --recurse-submodules https://github.com/Physical-Intelligence/openpi.git
  git -C openpi checkout -q 215abfb217dbac7d5f1273282331b9b1866c0479
  git -C openpi submodule update --init --recursive -q
fi

cd openpi
uv sync -q
uv pip install -q mujoco==3.12.0

# MuJoCo가 GPU로 렌더링하도록 EGL 설정
mkdir -p /usr/share/glvnd/egl_vendor.d
echo '{"file_format_version": "1.0.0", "ICD": {"library_path": "libEGL_nvidia.so.0"}}'   > /usr/share/glvnd/egl_vendor.d/10_nvidia.json

.venv/bin/python -c "import jax; print('JAX', jax.__version__, jax.devices())"
MUJOCO_GL=egl PYTHONPATH=/content/AIRLAB/ros2_ws/src/airlab_vla .venv/bin/python -c "
from airlab_vla.sim import PickScene
o = PickScene(seed=0, cameras='pi05').reset()
print('렌더링 확인', o['head'].shape, o['wrist'].shape, '평균 밝기', o['wrist'].mean().round(1))"

## 3. 데이터셋 풀기

Drive에서 바로 읽으면 느리므로, Colab의 로컬 디스크에 풉니다.

In [ ]:
%%bash
set -e
mkdir -p ~/.cache/huggingface/lerobot/airlab
time tar -xf "$DRIVE/pick_pi05.tar" -C ~/.cache/huggingface/lerobot/airlab
du -sh ~/.cache/huggingface/lerobot/airlab/pick_pi05

## 4. 학습

`pi05_base`(약 수 GB)를 받은 뒤 15,000 스텝을 학습합니다. 시작할 때 JAX 컴파일로 몇 분 멈춰 있는 것처럼 보일 수 있습니다.

처음 나오는 `Step 100`, `Step 200` 로그의 간격을 재면 전체 시간을 계산할 수 있습니다. 시간과 컴퓨팅 단위가 부담되면 중단하고 `--num-train-steps`를 추가해 줄이세요.

체크포인트는 5000 스텝마다 로컬에 저장되고, 뒤에서 도는 복사 작업이 Drive로 옮깁니다. Drive에는 가장 최근 것 하나만 남습니다.

In [ ]:
%%bash
cd /content/openpi
EXP=run1
LOCAL=/content/checkpoints/pi05_airlab/$EXP
SAVED=$DRIVE/checkpoints/pi05_airlab/$EXP
mkdir -p "$LOCAL" "$SAVED"

latest() { ls "$1" 2>/dev/null | grep -E '^[0-9]+$' | sort -n | tail -1; }
sync_drive() {
  s=$(latest "$LOCAL")
  if [ -n "$s" ] && [ ! -d "$SAVED/$s" ]; then
    find "$SAVED" -mindepth 1 -maxdepth 1 -exec rm -rf {} +
    cp -r "$LOCAL/$s" "$SAVED/$s.tmp" && mv "$SAVED/$s.tmp" "$SAVED/$s" && echo "Drive에 저장: step $s"
  fi
}

s=$(latest "$SAVED")
if [ -n "$s" ] && [ ! -d "$LOCAL/$s" ]; then
  echo "Drive의 step $s에서 이어서 학습합니다"
  cp -r "$SAVED/$s" "$LOCAL/$s"
fi

(while sleep 60; do sync_drive; done) &
SYNC=$!
# --resume: 체크포인트가 있으면 이어서, 빈 폴더면 처음부터 학습합니다
XLA_PYTHON_CLIENT_MEM_FRACTION=0.9 .venv/bin/python /content/AIRLAB/pi05/run_openpi.py scripts/train.py pi05_airlab   --exp-name $EXP --checkpoint-base-dir /content/checkpoints --batch-size $BATCH --resume
kill $SYNC
sync_drive
ls -la "$SAVED"

## 5. 평가

작은 정책과 같은 100개 시험 장면(시드 100000부터)에서 성공률을 잽니다. 결과는 Drive에 `eval100.txt`와 `eval.gif`로 저장됩니다.

In [ ]:
%%bash
cd /content/openpi
CKPT=$(ls -d /content/checkpoints/pi05_airlab/run1/[0-9]* "$DRIVE"/checkpoints/pi05_airlab/run1/[0-9]* 2>/dev/null | head -1)
echo "체크포인트: $CKPT"
MUJOCO_GL=egl XLA_PYTHON_CLIENT_MEM_FRACTION=0.6 .venv/bin/python /content/AIRLAB/pi05/evaluate_pi05.py   --checkpoint "$CKPT" --episodes 100 --video "$DRIVE/eval.gif" 2>&1 | tee "$DRIVE/eval100.txt" | tail -25

## 6. 언어 확인

두 상자가 있는 장면에서, 로봇이 정렬된 상자가 아니라 **옆 상자**의 색을 명령합니다. 작은 정책은 20번 모두 색 단어를 무시하고 정렬된 상자를 집었습니다(0/20). 데이터는 같으니 π0.5도 같은 지름길을 배웠을 수 있습니다.

In [ ]:
%%bash
cd /content/openpi
CKPT=$(ls -d /content/checkpoints/pi05_airlab/run1/[0-9]* "$DRIVE"/checkpoints/pi05_airlab/run1/[0-9]* 2>/dev/null | head -1)
MUJOCO_GL=egl XLA_PYTHON_CLIENT_MEM_FRACTION=0.6 .venv/bin/python /content/AIRLAB/pi05/evaluate_pi05.py   --checkpoint "$CKPT" --episodes 20 --ask-other 2>&1 | tee "$DRIVE/ask_other.txt" | tail -24